In [ ]:
import pyvisa as visa
from time import sleep
import matplotlib.pyplot as plt

import numpy as np
import time
from edes.modules.detection.detection_utils import plot, plot_ax, big_plt_font
import pandas as pd

import vxi11

In [ ]:

rm = visa.ResourceManager()
## uncomment the following if full address not known
instruments = rm.list_resources() 
instruments
rm

#usb = list(filter(lambda x: 'USB' in x, instruments))
#print(usb)
#PS = rm.open_resource('USB0::62700::5168::SPD3XJEQ6R3774::0::INSTR')


In [ ]:
import pyvisa
import time

def run_power_supply():

    rm = pyvisa.ResourceManager('@py')
    resource_name = 'USB0::62700::5168::SPD3XJEQ6R3774::0::INSTR'
    print(f"Connecting to: {resource_name}")

    try:
        # 3. Open the connection
        psu = rm.open_resource(resource_name)
        
        # Set a slight timeout (in milliseconds) just in case
        psu.timeout = 2000 

        # 4. Ask the device to identify itself
        idn = psu.query("*IDN?")
        print(f"\nSuccessfully connected to: {idn.strip()}")

        # 5. Configure Channel 1 (Safety first: 3.3 Volts, 0.5 Amps)
        print("Setting Channel 1 to 2V and 0.1A...")
        psu.write("CH1:VOLT 2")
        psu.write("CH1:CURR 0.1")
        
        # Wait a brief moment for the hardware to register the settings
        time.sleep(0.5) 

        # 6. Turn ON Channel 1 output
        print("Turning ON Channel 1...")
        psu.write("OUTP CH1,ON")
        
        # Give it a second to stabilize
        time.sleep(0.1)

        # 7. Read the actual output voltage and current
        actual_volt = psu.query("MEAS:VOLT? CH1")
        actual_curr = psu.query("MEAS:CURR? CH1")
        print(f"Measured Output -> Voltage: {actual_volt.strip()} V, Current: {actual_curr.strip()} A")

        # Let it run for 2 seconds so you can see the screen light up
        time.sleep(7)

        # 8. Turn OFF Channel 1 output
        print("Turning OFF Channel 1...")
        psu.write("OUTP CH1,OFF")
        print("Test complete!")

    except pyvisa.VisaIOError as e:
        print(f"\nCommunication error: {e}")
    finally:
        # 9. Clean up and close the connection
        if 'psu' in locals():
            psu.close()
            print("Connection closed.")

if __name__ == "__main__":
    for i in range(100):
    
        run_power_supply()
        time.sleep(15)

In [ ]:
import pyvisa
import time

def setup_rigol_pulses():
    # 1. Connect ONCE outside of any loops
    rm = pyvisa.ResourceManager('@py')
    inst = rm.open_resource('TCPIP0::192.168.169.68::INSTR')
    
    print("Connected to:", inst.query("*IDN?").strip())

    # 2. Basic Channel Setup
    inst.write(":OUTPut2 OFF")
    inst.write(":OUTPut2:LOAD INFinity")

    # 3. Use the built-in PULSE function instead of an arbitrary trace
    inst.write(":SOURce2:FUNCtion PULSe")

    # 4. Set the timing 
    # Period = 100 ms ON + 500 ms OFF = 600 ms total (0.6 seconds)
    inst.write(":SOURce2:PERiod 2")
    # Pulse Width = 100 ms (0.1 seconds)
    inst.write(":SOURce2:PULSe:WIDTh 0.01")

    # 5. Set Voltage (Amplitude: 20Vpp, Offset: 10V -> pulses from 0V to +20V)
    inst.write(":SOURce2:VOLTage:UNIT VPP")
    inst.write(":SOURce2:VOLTage 20")
    inst.write(":SOURce2:VOLTage:OFFSet 10")

    # 6. Configure Burst Mode (if you want EXACTLY 100 pulses)
    # If you want it to run forever, you can comment this block out.
    inst.write(":SOURce2:BURSt ON")
    inst.write(":SOURce2:BURSt:MODE TRIGgered")
    inst.write(":SOURce2:BURSt:NCYCles 1")        # Tell the hardware to stop after 100
    inst.write(":SOURce2:BURSt:TRIGger:SOURce MANual") # Wait for a software trigger from Python

    # 7. Turn the output on
    inst.write(":OUTPut2 ON")
    time.sleep(0.5) # Brief pause to let relays click over

    # 8. Fire the 100 pulses!
    print("Firing 100 pulses...")
    inst.write("*TRG") # This software trigger starts the burst
    
    # 100 pulses at 600ms each will take exactly 60 seconds to complete
    time.sleep(1)
    print("Burst complete.")
    
    inst.close()

if __name__ == "__main__":
    for i in range(100):
        
        setup_rigol_pulses()
        time.sleep(2)

In [13]:
import pyvisa
import time

def run_synchronized_test(oven_on_time):
    # 1. Initialize ONE Resource Manager for everything
    rm = pyvisa.ResourceManager('@py')
    psu = None
    awg = None

    try:
        # 2. Connect to both instruments
        print("Connecting to instruments...")
        psu = rm.open_resource('USB0::62700::5168::SPD3XJEQ6R3774::0::INSTR')
        awg = rm.open_resource('TCPIP0::192.168.169.50::INSTR')
        
        psu.timeout = 2000
        
        print(f"PSU: {psu.query('*IDN?').strip()}")
        print(f"AWG: {awg.query('*IDN?').strip()}")

        # 3. Pre-configure the Siglent PSU
        print("\nConfiguring PSU (2V, 0.1A)...")
        psu.write("CH1:VOLT 2")
        psu.write("CH1:CURR 1.7")
        psu.write("OUTP CH1,OFF") # Ensure it is strictly OFF before we begin

        # 4. Pre-configure the Rigol AWG
        print("Configuring AWG (10ms pulse, 20Vpp)...")
        awg.write(":OUTPut2 OFF")
        awg.write(":OUTPut2:LOAD INFinity")
        awg.write(":SOURce2:FUNCtion PULSe")
        awg.write(":SOURce2:PERiod 2")
        awg.write(":SOURce2:PULSe:WIDTh 0.01")
        awg.write(":SOURce2:VOLTage:UNIT VPP")
        awg.write(":SOURce2:VOLTage 10")
        awg.write(":SOURce2:VOLTage:OFFSet 10")
        
        # Setup burst waiting for a software trigger
        awg.write(":SOURce2:BURSt ON")
        awg.write(":SOURce2:BURSt:MODE TRIGgered")
        awg.write(":SOURce2:BURSt:NCYCles 1")
        awg.write(":SOURce2:BURSt:TRIGger:SOURce MANual")
        
        # Arm the AWG (It is ON, but outputting 0V until the trigger hits)
        awg.write(":OUTPut2 ON")
        
        # Give hardware relays and buffers a moment to settle
        time.sleep(0.5)

        # ---------------------------------------------------------
        # 5. THE SYNCHRONIZED EXECUTION
        # ---------------------------------------------------------
        print("\nFIRING BOTH DEVICES!")
        
        # We send these back-to-back. The PSU relay might take a few 
        # milliseconds to physically close, which usually perfectly 
        # aligns with the USB/LAN latency of the AWG trigger.
        psu.write("OUTP CH1,ON") 
        awg.write("*TRG")          
        # ---------------------------------------------------------
        
        # 6. Measure and Wait
        time.sleep(0.1) # Brief pause to let the PSU voltage stabilize
        actual_volt = psu.query("MEAS:VOLT? CH1").strip()
        actual_curr = psu.query("MEAS:CURR? CH1").strip()
        print(f"PSU Measured -> Voltage: {actual_volt} V, Current: {actual_curr} A")
        
        # Keep the script alive long enough for the 2-second AWG period to finish
        time.sleep(oven_on_time)
        psu.write("OUTP CH1,OFF")


    except pyvisa.VisaIOError as e:
        print(f"\nCommunication error: {e}")
        
    finally:
        # 7. Safe Teardown
        # This guarantees both devices turn off even if the code crashes
        print("\nCleaning up...")
        if psu:
            psu.write("OUTP CH1,OFF")
            psu.close()
        if awg:
            awg.write(":OUTPut2 OFF")
            awg.close()
        print("Connections closed. Test complete!")

if __name__ == "__main__":
    
    for i in range(3):
    
        run_synchronized_test(oven_on_time = 300)
        time.sleep(100)

Connecting to instruments...
PSU: Siglent Technologies,SPD3303X-E,SPD3XJEQ6R3774,1.01.01.02.07R2,V3.0
AWG: Rigol Technologies,DG4162,DG4E252901529,00.01.14

Configuring PSU (2V, 0.1A)...
Configuring AWG (10ms pulse, 20Vpp)...

FIRING BOTH DEVICES!
PSU Measured -> Voltage: 2.00 V, Current: 0.00 A

Cleaning up...
Connections closed. Test complete!
Connecting to instruments...
PSU: Siglent Technologies,SPD3303X-E,SPD3XJEQ6R3774,1.01.01.02.07R2,V3.0
AWG: Rigol Technologies,DG4162,DG4E252901529,00.01.14

Configuring PSU (2V, 0.1A)...
Configuring AWG (10ms pulse, 20Vpp)...

FIRING BOTH DEVICES!
PSU Measured -> Voltage: 2.00 V, Current: 0.00 A

Cleaning up...
Connections closed. Test complete!
Connecting to instruments...
PSU: Siglent Technologies,SPD3303X-E,SPD3XJEQ6R3774,1.01.01.02.07R2,V3.0
AWG: Rigol Technologies,DG4162,DG4E252901529,00.01.14

Configuring PSU (2V, 0.1A)...
Configuring AWG (10ms pulse, 20Vpp)...

FIRING BOTH DEVICES!
PSU Measured -> Voltage: 2.00 V, Current: 0.00 A

Cleani

In [10]:
rm.open_resource('TCPIP0::192.168.169.50::INSTR')

NameError: name 'rm' is not defined

In [ ]:
rm = pyvisa.ResourceManager('@py')